# Fit the Required Scaling Parameters in the Analytical "Surrogate Models"

Works in https://doi.org/10.1002/jbio.202300536 are limited to the visible ranges:

> Analytical models can be applied to ranges of wavelengths for which they are developed and so these are favored in this work. Here we consider only visible range models (450–650 nm) as this allows for non-contact imaging using standard cold surgical light sources [11].
- [Bahl, A., Segaud, S., Xie, Y., Shapey, J., Bergholt, M. S., & Vercauteren, T. (2024). A comparative study of analytical models of diffuse reflectance in homogeneous biological tissues: Gelatin‐based phantoms and Monte Carlo experiments. Journal of biophotonics, 17(6), e202300536.](https://doi.org/10.1002/jbio.202300536)

Original works by Jacques et al. (1999) and Yudovsky et al. (2009) do not explicitly state those wavelength limitations.

## Comparison of Fitting Parameters

To compare obtained fitting results with previous results, please see Table 2 in https://doi.org/10.1002/jbio.202300536.

In [ ]:
import os
from math import inf

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from scipy.optimize import curve_fit

from mcmlnet.susi.calculate_spectra import calculate_spectrum_for_physical_batch
from mcmlnet.training.optimization.analytical_baselines import (
    AnalyticalModelConstants,
    compute_mu_a,
    compute_mu_s_prime,
    jacques_1999,
    modified_beer_lambert,
    yudovsky_2009,
)
from mcmlnet.utils.metrics import compute_and_report_fit_quality

load_dotenv()

%matplotlib inline

## Simulate Physiological Reflectance Data to Fit Models On

In [ ]:
# Configuration
result_dir = os.path.join(
    os.getenv("data_dir"),
    "raw/related_work_reimplemented/",
)
consts = AnalyticalModelConstants()
wavelengths = consts.wavelengths
n_photons = consts.n_photons
d = consts.thickness
n_sims = consts.n_sims
n_layers = 1

results = []
refractive_indices = [1.33, 1.35, 1.44]

# Create result directory if it does not exist
for iteration, n in enumerate(refractive_indices):
    try:
        result_df = pd.read_parquet(
            os.path.join(result_dir, f"jacques_1999_n_{n}_resim.parquet")
        )
    except FileNotFoundError:
        # Sample data
        np.random.seed(0)
        sao2 = np.random.uniform(consts.sao2_range[0], consts.sao2_range[1], n_sims)
        np.random.seed(1)
        f_blood = np.random.uniform(
            consts.f_blood_range[0], consts.f_blood_range[1], n_sims
        )
        np.random.seed(2)
        a_mie = np.random.uniform(consts.a_mie_range[0], consts.a_mie_range[1], n_sims)
        np.random.seed(3)
        b_mie = np.random.uniform(consts.b_mie_range[0], consts.b_mie_range[1], n_sims)
        np.random.seed(4)
        g = np.random.uniform(consts.g_range[0], consts.g_range[1], n_sims)
        n_array = np.ones(n_sims) * n
        d_array = np.ones(n_sims) * d

        # Repeat uniform data across all wavelengths
        g = np.repeat(g[:, np.newaxis], len(wavelengths), axis=1)
        n_array = np.repeat(n_array[:, np.newaxis], len(wavelengths), axis=1)
        d_array = np.repeat(d_array[:, np.newaxis], len(wavelengths), axis=1)

        # Compute absorption and scattering coefficients
        mu_a = compute_mu_a(sao2, f_blood, wavelengths)
        mu_s = compute_mu_s_prime(a_mie, b_mie, wavelengths) / (1 - g)

        # Stack data
        data_batch = np.stack([mu_a, mu_s, g, n_array, d_array], axis=-1).reshape(
            len(mu_a), -1
        )

        # Create the parameter DataFrame for simulation
        header = pd.MultiIndex.from_product(
            [
                [f"layer{i}" for i in range(n_layers)],
                np.round(wavelengths, 12).tolist(),
                ["ua", "us", "g", "n", "d"],
            ],
            names=["layer [top first]", "wavelength [m]", "parameter"],
        )
        result_df = pd.DataFrame(data_batch, columns=header)

        # Compute the reflectance
        reflectance = calculate_spectrum_for_physical_batch(
            batch=result_df,
            wavelengths=wavelengths,  # dummy value
            nr_photons=n_photons,
            mci_base_folder="",
            batch_id=str(iteration),
            ignore_a=True,
            mco_file="batch.mco",
        )
        # Adapt and append the reflectance DataFrame
        result_header = pd.MultiIndex.from_product(
            [
                ["reflectance"],
                wavelengths.tolist(),
                [""],
            ],
        )
        adapted_reflectance_df = pd.DataFrame(
            reflectance.to_numpy(), columns=result_header
        )
        result_df = pd.concat([result_df, adapted_reflectance_df], axis=1)
        result_df.to_parquet(
            os.path.join(result_dir, f"jacques_1999_n_{n}_resim.parquet"), index=False
        )
    results.append(result_df)

# Get min and max values for each physical parameter across all simulations
idx = pd.IndexSlice
min_max_values = {}
parameters = ["ua", "us", "g", "n", "d"]

for param in parameters:
    sliced_df = result_df.loc[:, idx[:, :, param]]
    min_value = sliced_df.min().min()
    max_value = sliced_df.max().max()
    min_max_values[param] = {"min": min_value, "max": max_value}

min_max_df = pd.DataFrame(min_max_values).T
print(min_max_df, results[0].shape)

In [ ]:
# Plot example reflectance spectra
plt.figure(figsize=(10, 5))
for i in range(5):
    plt.plot(wavelengths * 1e9, results[0].iloc[i]["reflectance"], color="tab:blue")
    plt.plot(wavelengths * 1e9, results[1].iloc[i]["reflectance"], color="tab:orange")
    plt.plot(wavelengths * 1e9, results[2].iloc[i]["reflectance"], color="tab:green")
plt.xlabel("Wavelength [nm]")
plt.ylabel("Reflectance")
plt.title("Example reflectance spectra for different refractive indices")
plt.legend(
    handles=[
        plt.Line2D([0], [0], color="tab:blue", label="n=1.33"),
        plt.Line2D([0], [0], color="tab:orange", label="n=1.35"),
        plt.Line2D([0], [0], color="tab:green", label="n=1.44"),
    ]
)
plt.show()

In [ ]:
# define the general purpose fitting function
def fit_model_to_data(
    mu_a: np.ndarray,
    mu_s_prime: np.ndarray,
    diffuse_reflectance_data: np.ndarray,
    model_function: callable,
    initial_guess: list[float],
    bounds: tuple[list[float], list[float]] = (-inf, inf),
    **model_kwargs,
):
    """
    Fit the parameters M1, M2, and M3 of the modified Beer-Lambert law
    to the given data.

    Args:
        mu_a: np.ndarray, absorption coefficient [1/m]
        mu_s_prime: np.ndarray, reduced scattering coefficient [1/m]
        diffuse_reflectance_data: np.ndarray, target diffuse reflectance data
        model_function: callable, the model function to fit
        initial_guess: list[float], initial guess for the model parameters
        bounds: tuple[list[float], list[float]], bounds for the model parameters
        model_kwargs: additional keyword arguments for the model function

    Returns
        dict[str, float], fitted parameters M1-M3 or M1-M6
    """
    assert mu_a.shape == mu_s_prime.shape == diffuse_reflectance_data.shape
    assert mu_a.ndim == mu_s_prime.ndim == diffuse_reflectance_data.ndim == 2

    inputs = (mu_a.reshape(-1), mu_s_prime.reshape(-1))

    # fit the model function to the data
    popt, pcov = curve_fit(
        # model_function,
        lambda inputs, *params: model_function(inputs, *params, **model_kwargs),
        inputs,
        diffuse_reflectance_data.reshape(-1),
        p0=initial_guess,
        bounds=bounds,
        maxfev=100000,
    )

    # return the fitted parameters
    _names = [f"M{i}" for i in range(1, len(initial_guess) + 1)]
    fitted_params = {name: round(float(popt[i]), 4) for i, name in enumerate(_names)}
    fitted_uncertainties = {
        name: round(float(np.sqrt(pcov[i, i])), 4) for i, name in enumerate(_names)
    }
    print(f"Fitted parameters: {fitted_params}")
    print(f"Fitted uncertainties: {fitted_uncertainties}")

    # compute and report the fit quality
    predictions = model_function(inputs, *popt, **model_kwargs).reshape(
        diffuse_reflectance_data.shape
    )
    compute_and_report_fit_quality(
        predictions, diffuse_reflectance_data, model_function.__name__
    )

    return fitted_params, fitted_uncertainties

## Beer-Lambert

### Fit the Beer-Lambert model to the training data

In [ ]:
# define the model function for modified Beer-Lambert law
def model_function_lambert_beer(
    inputs: tuple[np.ndarray], M1: float, M2: float, M3: float
):
    mu_a, mu_s_prime = inputs
    params = {"M1": M1, "M2": M2, "M3": M3}
    return modified_beer_lambert(mu_a, mu_s_prime, params)

In [ ]:
for _i, result in enumerate(results):
    # extract the data for the fitting
    idx = pd.IndexSlice
    mu_a = result.loc[:, idx[:, :, "ua"]].to_numpy()
    mu_s_prime = result.loc[:, idx[:, :, "us"]].to_numpy() * (
        1 - result.loc[:, idx[:, :, "g"]].to_numpy()
    )
    diffuse_reflectance_data = result["reflectance"].to_numpy()

    # fit the parameters
    initial_guess = [0.1, 0.1, 0.1]
    params, _ = fit_model_to_data(
        mu_a,
        mu_s_prime,
        diffuse_reflectance_data,
        model_function_lambert_beer,
        initial_guess,
    )

In [ ]:
# plot the results
def plot_simulation_and_model(
    result: pd.DataFrame,
    wavelengths: np.ndarray,
    params: dict[str, float],
    iteration: int,
    model: callable,
    **kwargs,
) -> None:
    plt.figure(figsize=(10, 5))
    for i in range(5):
        plt.plot(wavelengths * 1e9, result.iloc[i]["reflectance"], color="tab:blue")
        # plot the fitted model
        mu_a = result.loc[:, idx[:, :, "ua"]].to_numpy()
        mu_s_prime = result.loc[:, idx[:, :, "us"]].to_numpy() * (
            1 - result.loc[:, idx[:, :, "g"]].to_numpy()
        )
        surrogate_reflectance = model(mu_a, mu_s_prime, params, **kwargs)
        plt.plot(
            wavelengths * 1e9,
            surrogate_reflectance[i],
            color="tab:orange",
            linestyle="--",
        )
    plt.xlabel("Wavelength [nm]")
    plt.ylabel("Reflectance")
    plt.title(
        "Example reflectance spectra for refractive index "
        f"n={refractive_indices[iteration]}"
    )
    plt.legend(
        handles=[
            plt.Line2D(
                [0], [0], color="tab:blue", label=f"n={refractive_indices[iteration]}"
            ),
            plt.Line2D(
                [0],
                [0],
                color="tab:orange",
                label=f"n={refractive_indices[iteration]} (fitted)",
            ),
        ]
    )
    plt.show()


for id, result in enumerate(results):
    plot_simulation_and_model(result, wavelengths, params, id, modified_beer_lambert)

### Fit the Beer-Lambert model to the training data, including specular reflectance

In [ ]:
def r_specular(n1: float, n2: float) -> float:
    """
    Computes the specular reflectance to be added to the default MCML reflectance.

    Args:
        n1: refractive index of the top medium/ air
        n2: refractive index of the bottom medium/ first tissue layer

    Returns:
        The specular reflectance value.
    """
    return (n1 - n2) ** 2 / (n1 + n2) ** 2


for i, result in enumerate(results):
    # extract the data for the fitting
    idx = pd.IndexSlice
    mu_a = result.loc[:, idx[:, :, "ua"]].to_numpy()
    mu_s_prime = result.loc[:, idx[:, :, "us"]].to_numpy() * (
        1 - result.loc[:, idx[:, :, "g"]].to_numpy()
    )
    diffuse_reflectance_data = result["reflectance"].to_numpy() + r_specular(
        1.0, refractive_indices[i]
    )

    # fit the parameters
    initial_guess = [0.1, 0.1, 0.1]
    bounds = (0, inf)
    params, _ = fit_model_to_data(
        mu_a,
        mu_s_prime,
        diffuse_reflectance_data,
        model_function_lambert_beer,
        initial_guess,
        bounds=bounds,
    )

In [ ]:
# plot the results
for id, result in enumerate(results):
    plot_simulation_and_model(result, wavelengths, params, id, modified_beer_lambert)

### Fit the Beer-Lambert model to the training data, only positive parameters

In [ ]:
for _i, result in enumerate(results):
    # extract the data for the fitting
    idx = pd.IndexSlice
    mu_a = result.loc[:, idx[:, :, "ua"]].to_numpy()
    mu_s_prime = result.loc[:, idx[:, :, "us"]].to_numpy() * (
        1 - result.loc[:, idx[:, :, "g"]].to_numpy()
    )
    diffuse_reflectance_data = result["reflectance"].to_numpy()

    # fit the parameters
    initial_guess = [0.1, 0.1, 0.1]
    bounds = (0, inf)
    params, _ = fit_model_to_data(
        mu_a,
        mu_s_prime,
        diffuse_reflectance_data,
        model_function_lambert_beer,
        initial_guess,
        bounds=bounds,
    )

In [ ]:
# plot the results
for id, result in enumerate(results):
    plot_simulation_and_model(result, wavelengths, params, id, modified_beer_lambert)

### Fit the Beer-Lambert model to the training data, limit wavelength range to 450-650 nm

In [ ]:
# define bool mask to access the data in range [450, 650] nm
restricted_wvls = (wavelengths >= 450e-9) & (wavelengths <= 650e-9)

In [ ]:
# further constrain the parameters, closer to the literature values
for _i, result in enumerate(results):
    # extract the data for the fitting
    idx = pd.IndexSlice
    mu_a = result.loc[:, idx[:, :, "ua"]].to_numpy()
    mu_s_prime = result.loc[:, idx[:, :, "us"]].to_numpy() * (
        1 - result.loc[:, idx[:, :, "g"]].to_numpy()
    )
    diffuse_reflectance_data = result["reflectance"].to_numpy()

    # fit the parameters
    initial_guess = [0.1, 0.1, 0.1]
    params, _ = fit_model_to_data(
        mu_a[:, restricted_wvls],
        mu_s_prime[:, restricted_wvls],
        diffuse_reflectance_data[:, restricted_wvls],
        model_function_lambert_beer,
        initial_guess,
    )

In [ ]:
# plot the results
for id, result in enumerate(results):
    plot_simulation_and_model(result, wavelengths, params, id, modified_beer_lambert)

## Jacques 1999

In [ ]:
# define the model function for Jacques 1999
def model_function_jacques_1999(
    inputs: tuple[np.ndarray], M1: float, M2: float, M3: float
):
    mu_a, mu_s_prime = inputs
    params = {"M1": M1, "M2": M2, "M3": M3}
    return jacques_1999(mu_a, mu_s_prime, params)

### Fit the Jacques 1999 model to the training data

In [ ]:
for _i, result in enumerate(results):
    # extract the data for the fitting
    idx = pd.IndexSlice
    mu_a = result.loc[:, idx[:, :, "ua"]].to_numpy()
    mu_s_prime = result.loc[:, idx[:, :, "us"]].to_numpy() * (
        1 - result.loc[:, idx[:, :, "g"]].to_numpy()
    )
    diffuse_reflectance_data = result["reflectance"].to_numpy()

    # fit the parameters
    initial_guess = [0.1, 0.1, 0.1]
    params, _ = fit_model_to_data(
        mu_a,
        mu_s_prime,
        diffuse_reflectance_data,
        model_function_jacques_1999,
        initial_guess,
    )

In [ ]:
# plot the results
for id, result in enumerate(results):
    plot_simulation_and_model(result, wavelengths, params, id, jacques_1999)

### Fit the Jacques 1999 model to the training data, including specular reflectance

In [ ]:
for i, result in enumerate(results):
    # extract the data for the fitting
    idx = pd.IndexSlice
    mu_a = result.loc[:, idx[:, :, "ua"]].to_numpy()
    mu_s_prime = result.loc[:, idx[:, :, "us"]].to_numpy() * (
        1 - result.loc[:, idx[:, :, "g"]].to_numpy()
    )
    diffuse_reflectance_data = result["reflectance"].to_numpy() + r_specular(
        1.0, refractive_indices[i]
    )

    # fit the parameters
    initial_guess = [0.1, 0.1, 0.1]
    params, _ = fit_model_to_data(
        mu_a,
        mu_s_prime,
        diffuse_reflectance_data,
        model_function_jacques_1999,
        initial_guess,
    )

In [ ]:
# plot the results
for id, result in enumerate(results):
    plot_simulation_and_model(result, wavelengths, params, id, jacques_1999)

### Fit the Jacques 1999 model to the training data, limit wavelength range to 450-650 nm

In [ ]:
for _i, result in enumerate(results):
    # extract the data for the fitting
    idx = pd.IndexSlice
    mu_a = result.loc[:, idx[:, :, "ua"]].to_numpy()
    mu_s_prime = result.loc[:, idx[:, :, "us"]].to_numpy() * (
        1 - result.loc[:, idx[:, :, "g"]].to_numpy()
    )
    diffuse_reflectance_data = result["reflectance"].to_numpy()

    # fit the parameters
    initial_guess = [0.1, 0.1, 0.1]
    params, _ = fit_model_to_data(
        mu_a[:, restricted_wvls],
        mu_s_prime[:, restricted_wvls],
        diffuse_reflectance_data[:, restricted_wvls],
        model_function_jacques_1999,
        initial_guess,
    )

In [ ]:
# plot the results
for id, result in enumerate(results):
    plot_simulation_and_model(result, wavelengths, params, id, jacques_1999)

## Yudovsky 2009

In [ ]:
# define the model function for Yudovsky 2009
def model_function_yudovsky_2009(
    inputs: tuple[np.ndarray],
    M1: float,
    M2: float,
    M3: float,
    M4: float,
    M5: float,
    M6: float,
    kind: str = "original",
):
    mu_a, mu_s_prime = inputs
    params = {"M1": M1, "M2": M2, "M3": M3, "M4": M4, "M5": M5, "M6": M6}
    return yudovsky_2009(mu_a, mu_s_prime, params, kind=kind)

### Fit the Yudovsky 2009 model to the training data

In [ ]:
for _i, result in enumerate(results):
    # extract the data for the fitting
    idx = pd.IndexSlice
    mu_a = result.loc[:, idx[:, :, "ua"]].to_numpy()
    mu_s_prime = result.loc[:, idx[:, :, "us"]].to_numpy() * (
        1 - result.loc[:, idx[:, :, "g"]].to_numpy()
    )
    diffuse_reflectance_data = result["reflectance"].to_numpy()

    # fit the parameters
    initial_guess = [0.1, 0.1, 0.1, 0.1, 0.1, 0.1]
    params, _ = fit_model_to_data(
        mu_a,
        mu_s_prime,
        diffuse_reflectance_data,
        model_function_yudovsky_2009,
        initial_guess,
        kind="original",
    )

In [ ]:
# plot the results
for id, result in enumerate(results):
    plot_simulation_and_model(
        result, wavelengths, params, id, yudovsky_2009, kind="original"
    )

### Fit the Yudovsky 2009 model to the training data, including specular reflectance

In [ ]:
for i, result in enumerate(results):
    # extract the data for the fitting
    idx = pd.IndexSlice
    mu_a = result.loc[:, idx[:, :, "ua"]].to_numpy()
    mu_s_prime = result.loc[:, idx[:, :, "us"]].to_numpy() * (
        1 - result.loc[:, idx[:, :, "g"]].to_numpy()
    )
    diffuse_reflectance_data = result["reflectance"].to_numpy() + r_specular(
        1.0, refractive_indices[i]
    )

    # fit the parameters
    initial_guess = [0.1, 0.1, 0.1, 0.1, 0.1, 0.1]
    params, _ = fit_model_to_data(
        mu_a,
        mu_s_prime,
        diffuse_reflectance_data,
        model_function_yudovsky_2009,
        initial_guess,
        kind="original",
    )

In [ ]:
# plot the results
for id, result in enumerate(results):
    plot_simulation_and_model(
        result, wavelengths, params, id, yudovsky_2009, kind="original"
    )

### Fit the Yudovsky 2009 model to the training data, using the original implementation from the Erratum

In [ ]:
for _i, result in enumerate(results):
    # extract the data for the fitting
    idx = pd.IndexSlice
    mu_a = result.loc[:, idx[:, :, "ua"]].to_numpy()
    mu_s_prime = result.loc[:, idx[:, :, "us"]].to_numpy() * (
        1 - result.loc[:, idx[:, :, "g"]].to_numpy()
    )
    diffuse_reflectance_data = result["reflectance"].to_numpy()

    # fit the parameters
    initial_guess = [0.1, 0.1, 0.1, 0.1, 0.1, 0.1]
    bounds = (0, inf)
    params, _ = fit_model_to_data(
        mu_a,
        mu_s_prime,
        diffuse_reflectance_data,
        model_function_yudovsky_2009,
        initial_guess,
        bounds=bounds,
        kind="erratum",
    )

In [ ]:
# plot the results
for id, result in enumerate(results):
    plot_simulation_and_model(
        result, wavelengths, params, id, yudovsky_2009, kind="erratum"
    )

### Fit the Yudovsky 2009 model to the training data, using the implementation from Bahl et al. (2024)

In [ ]:
for _i, result in enumerate(results):
    # extract the data for the fitting
    idx = pd.IndexSlice
    mu_a = result.loc[:, idx[:, :, "ua"]].to_numpy()
    mu_s_prime = result.loc[:, idx[:, :, "us"]].to_numpy() * (
        1 - result.loc[:, idx[:, :, "g"]].to_numpy()
    )
    diffuse_reflectance_data = result["reflectance"].to_numpy()

    # fit the parameters
    initial_guess = [0.1, 0.1, 0.1, 0.1, 0.1, 0.1]
    params, _ = fit_model_to_data(
        mu_a[:, restricted_wvls],
        mu_s_prime[:, restricted_wvls],
        diffuse_reflectance_data[:, restricted_wvls],
        model_function_yudovsky_2009,
        initial_guess,
        kind="bahl_reproduced",
    )

In [ ]:
# plot the results
for id, result in enumerate(results):
    plot_simulation_and_model(
        result, wavelengths, params, id, yudovsky_2009, kind="bahl_reproduced"
    )